In [1]:
import os
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

jar_path = f"{os.getcwd()}/postgresql-42.7.0.jar"

spark = (SparkSession.builder.appName("Silver-Layer")
    .config("spark.jars", jar_path)
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate())

print("Spark:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

BRONZE = "/home/jovyan/work/data/bronze"
SILVER = "/home/jovyan/work/data/silver"

Spark: 3.5.0
Spark UI: http://a41fb29cf9be:4041


In [2]:
orders_b = spark.read.parquet(f"{BRONZE}/orders")
order_products_b = spark.read.parquet(f"{BRONZE}/order_products_prior")
products_b = spark.read.parquet(f"{BRONZE}/products")
aisles_b = spark.read.parquet(f"{BRONZE}/aisles")
departments_b = spark.read.parquet(f"{BRONZE}/departments")

print("=== Bronze Row Counts ===")
print(f"orders: {orders_b.count():,}")
print(f"order_products_prior: {order_products_b.count():,}")
print(f"products: {products_b.count():,}")
print(f"aisles: {aisles_b.count():,}")
print(f"departments: {departments_b.count():,}")

print("\n=== NULL Check (orders) ===")
orders_b.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) 
    for c in orders_b.columns
]).show()

=== Bronze Row Counts ===
orders: 3,421,083
order_products_prior: 32,434,489
products: 49,688
aisles: 134
departments: 21

=== NULL Check (orders) ===
+--------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+-------+--------+------------+---------+-----------------+----------------------+
|       0|      0|       0|           0|        0|                0|                206209|
+--------+-------+--------+------------+---------+-----------------+----------------------+



In [3]:
dim_products = (products_b
    .join(F.broadcast(aisles_b), "aisle_id", "left")
    .join(F.broadcast(departments_b), "department_id", "left")
    .select(
        F.col("product_id"),
        F.trim(F.col("product_name")).alias("product_name"),
        F.col("aisle_id"),
        F.col("aisle"),
        F.col("department_id"),
        F.col("department"),
        F.current_timestamp().alias("silver_loaded_at")
    )
    .filter(F.col("product_name").isNotNull())
    .dropDuplicates(["product_id"]))

(dim_products.write
    .format("delta")
    .mode("overwrite")
    .save(f"{SILVER}/dim_products"))

print(f"✅ dim_products: {dim_products.count():,} rows written as Delta")
dim_products.show(5, truncate=False)

✅ dim_products: 49,688 rows written as Delta
+----------+----------------------------------------------+--------+-----------------------------+-------------+-------------+-----------------------+
|product_id|product_name                                  |aisle_id|aisle                        |department_id|department   |silver_loaded_at       |
+----------+----------------------------------------------+--------+-----------------------------+-------------+-------------+-----------------------+
|5         |Green Chile Anytime Sauce                     |5       |marinades meat preparation   |13           |pantry       |2026-05-01 20:34:49.023|
|6         |Dry Nose Oil                                  |11      |cold flu allergy             |11           |personal care|2026-05-01 20:34:49.023|
|9         |Light Strawberry Blueberry Yogurt             |120     |yogurt                       |16           |dairy eggs   |2026-05-01 20:34:49.023|
|10        |Sparkling Orange Juice & Prickly Pear

In [4]:
dim_users = (orders_b
    .groupBy("user_id")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.min("order_number").alias("first_order_number"),
        F.max("order_number").alias("last_order_number"),
        F.round(F.avg("days_since_prior_order"), 2).alias("avg_days_between_orders"),
        F.current_timestamp().alias("silver_loaded_at")
    ))

(dim_users.write
    .format("delta")
    .mode("overwrite")
    .save(f"{SILVER}/dim_users"))

print(f"✅ dim_users: {dim_users.count():,} rows written as Delta")
dim_users.show(5)

✅ dim_users: 206,209 rows written as Delta
+-------+------------+------------------+-----------------+-----------------------+--------------------+
|user_id|total_orders|first_order_number|last_order_number|avg_days_between_orders|    silver_loaded_at|
+-------+------------+------------------+-----------------+-----------------------+--------------------+
|     12|           6|                 1|                6|                   26.0|2026-05-01 20:34:...|
|     13|          13|                 1|               13|                   7.67|2026-05-01 20:34:...|
|     14|          14|                 1|               14|                  21.23|2026-05-01 20:34:...|
|     38|          13|                 1|               13|                  21.75|2026-05-01 20:34:...|
|     46|          20|                 1|               20|                  17.37|2026-05-01 20:34:...|
+-------+------------+------------------+-----------------+-----------------------+--------------------+
only showing

In [5]:
fact_order_items = (order_products_b
    .filter(F.col("order_id").isNotNull() & F.col("product_id").isNotNull())
    .dropDuplicates(["order_id", "product_id"])
    .join(orders_b.select("order_id", "user_id", "order_dow", 
                          "order_hour_of_day", "days_since_prior_order"),
          on="order_id", how="inner")
    .join(F.broadcast(dim_products.select("product_id", "department", "aisle")),
          on="product_id", how="left")
    .withColumn("silver_loaded_at", F.current_timestamp())
    .select(
        "order_id", "user_id", "product_id",
        "department", "aisle",
        "add_to_cart_order", "reordered",
        "order_dow", "order_hour_of_day", "days_since_prior_order",
        "silver_loaded_at"
    ))

(fact_order_items.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("department")
    .save(f"{SILVER}/fact_order_items"))

print("✅ fact_order_items written (partitioned by department)")

✅ fact_order_items written (partitioned by department)


In [7]:
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, f"{SILVER}/fact_order_items")

# Delta-only feature: table history
print("=== Delta Table History ===")
dt.history().select("version", "timestamp", "operation", 
                    "operationMetrics.numOutputRows").show(truncate=False)

# Verify partition structure
print("\n=== Partitions on disk ===")
import subprocess
result = subprocess.run(
    ["ls", f"{SILVER}/fact_order_items"], 
    capture_output=True, text=True
)
print(result.stdout)

# Final row count
fact_check = spark.read.format("delta").load(f"{SILVER}/fact_order_items")
print(f"\n=== Silver fact_order_items final count: {fact_check.count():,} ===")

=== Delta Table History ===
+-------+-----------------------+---------+-------------+
|version|timestamp              |operation|numOutputRows|
+-------+-----------------------+---------+-------------+
|0      |2026-05-01 20:35:32.093|WRITE    |32434489     |
+-------+-----------------------+---------+-------------+


=== Partitions on disk ===
_delta_log
department=alcohol
department=babies
department=bakery
department=beverages
department=breakfast
department=bulk
department=canned goods
department=dairy eggs
department=deli
department=dry goods pasta
department=frozen
department=__HIVE_DEFAULT_PARTITION__
department=household
department=international
department=meat seafood
department=missing
department=other
department=pantry
department=personal care
department=pets
department=produce
department=snacks


=== Silver fact_order_items final count: 32,434,489 ===
